In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from keras import models,layers
from tensorflow.keras.applications import MobileNetV3Small
from tensorflow.keras.applications.mobilenet_v3 import preprocess_input
from tensorflow.keras.callbacks import ReduceLROnPlateau
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import classification_report
import matplotlib.pyplot as plt
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
dataset_training = np.load("")
X_train = dataset_training["X_train"]
y_train = dataset_training["y_train"]

In [ ]:
dataset_testing = np.load("")
X_test = dataset_testing["X_test"]
y_test = dataset_testing["y_test"]

In [ ]:
values_train ,counts_train = np.unique(y_train,return_counts=True)
values_train,counts_train

(array([0, 1]), array([1000, 1000]))

In [ ]:
values_test ,counts_test = np.unique(y_test,return_counts=True)
values_test,counts_test

(array([0, 1]), array([ 62, 452]))

In [ ]:
X_train_preprocessed = preprocess_input(X_train.copy())
X_test_preprocessed = preprocess_input(X_test.copy())

In [ ]:
num_classes = 1
base_model = MobileNetV3Small(
    weights="imagenet",
    include_top=False,
    input_shape=(224,224, 3),
)

4334752/4334752 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
base_model.trainable = False

In [ ]:
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation="relu",
    ),
    layers.Dropout(0.5),
    layers.Dense(num_classes, activation="sigmoid")
])


In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
reduce_lr = ReduceLROnPlateau(
    monitor='val_accuracy', factor=0.5, patience=3, min_lr=1e-10, verbose=1
)
early_stop = EarlyStopping(
    monitor='val_accuracy', patience=5, restore_best_weights=True,verbose=1
)

In [ ]:
history1 = model.fit(
    X_train_preprocessed,
    y_train,
    epochs=20,
    callbacks=[reduce_lr,early_stop],
    validation_data=(X_test_preprocessed,y_test),
    shuffle=True,
)

Epoch 1/20
63/63 ━━━━━━━━━━━━━━━━━━━━ 34s 312ms/step - accuracy: 0.9240 - loss: 0.1834 - val_accuracy: 0.9767 - val_loss: 0.0617 - learning_rate: 0.0010
Epoch 2/20
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - accuracy: 0.9765 - loss: 0.0711 - val_accuracy: 0.9669 - val_loss: 0.0905 - learning_rate: 0.0010
Epoch 3/20
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.9875 - loss: 0.0422 - val_accuracy: 0.9650 - val_loss: 0.0914 - learning_rate: 0.0010
Epoch 4/20
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - accuracy: 0.9865 - loss: 0.0378 - val_accuracy: 0.9825 - val_loss: 0.0460 - learning_rate: 0.0010
Epoch 5/20
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.9910 - loss: 0.0282 - val_accuracy: 0.9805 - val_loss: 0.0546 - learning_rate: 0.0010
Epoch 6/20
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.9940 - loss: 0.0239 - val_accuracy: 0.9689 - val_loss: 0.0690 - learning_rate: 0.0010
Epoch 7/20
61/63 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.9976 - loss: 0.0145
Epoch 7

In [ ]:
# Get probabilities
y_pred_prob = model.predict(X_test_preprocessed)

# Convert probabilities to 0 or 1
y_pred = (y_pred_prob >= 0.5).astype(int).ravel()

# Classification report
print(classification_report(
    y_test,
    y_pred,
    target_names=["Healthy", "Unhealthy"]
))

17/17 ━━━━━━━━━━━━━━━━━━━━ 9s 274ms/step
              precision    recall  f1-score   support

     Healthy       0.90      1.00      0.95        62
   Unhealthy       1.00      0.98      0.99       452

    accuracy                           0.99       514
   macro avg       0.95      0.99      0.97       514
weighted avg       0.99      0.99      0.99       514



In [ ]:
import json

with open("nails_disease_binary_classifier_model_history.json", "w") as f:
    json.dump(history1.history, f)

In [ ]:
with open("nails_disease_binary_classifier_model_history.json", "r") as f:
    history_dict = json.load(f)

In [ ]:
model.save("nails_disease_binary_classification_model.keras")